In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt

from scipy.optimize import minimize_scalar
from scipy.special import expit, logit
from scipy.stats import norm
# import utilities as utils

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GaussianNetwork(nn.Module):
    """
    Neural network mapping x -> parameters of an N_y-dimensional Gaussian.

    Outputs
    -------
    mean : Tensor, shape (..., N_y)
        Gaussian mean.

    scale_tril : Tensor, shape (..., N_y, N_y)
        Lower-triangular Cholesky factor L such that
            covariance = L @ L.T

    covariance : Tensor, shape (..., N_y, N_y)
        Gaussian covariance matrix.
    """

    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_dims=(128, 128),
        min_std=1e-4,
    ):
        super().__init__()

        self.output_dim = output_dim
        self.min_std = min_std

        # Shared feature-processing network
        layers = []
        d = input_dim

        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(d, hidden_dim),
                nn.ReLU(),
                nn.Dropout(p=0.1),
            ])
            d = hidden_dim

        self.backbone = nn.Sequential(*layers)

        # Mean requires N_y parameters
        self.mean_head = nn.Linear(d, output_dim)

        # Lower triangular matrix requires
        # N_y * (N_y + 1) / 2 parameters
        n_tril = output_dim * (output_dim + 1) // 2
        self.cholesky_head = nn.Linear(d, n_tril)

        # Indices of lower-triangular matrix elements
        tril_indices = torch.tril_indices(
            row=output_dim,
            col=output_dim,
            offset=0,
        )

        self.register_buffer("tril_indices", tril_indices)


    def forward(self, x):
        h = self.backbone(x)

        # Mean
        mean = self.mean_head(h)

        # Raw parameters for Cholesky factor
        raw_tril = self.cholesky_head(h)

        batch_shape = x.shape[:-1]

        L = torch.zeros(
            *batch_shape,
            self.output_dim,
            self.output_dim,
            device=x.device,
            dtype=x.dtype,
        )

        L[..., self.tril_indices[0], self.tril_indices[1]] = raw_tril

        # Diagonal must be positive.
        diag_idx = torch.arange(self.output_dim, device=x.device)

        raw_diag = L[..., diag_idx, diag_idx]

        L[..., diag_idx, diag_idx] = (
            F.softplus(raw_diag) + self.min_std
        )

        # covariance = L @ L.transpose(-1, -2)

        # return mean, L, covariance
        return mean, L

class GRUGaussianNetwork(nn.Module):
    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_dim=64,
        num_layers=1,
        head_hidden_dims=(128,),
        min_std=1e-4,
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
        )

        self.gaussian_head = GaussianNetwork(
            input_dim=hidden_dim,
            output_dim=output_dim,
            hidden_dims=head_hidden_dims,
            min_std=min_std,
        )

    def forward(self, x, h=None):
        # x: (batch, time, input_dim)
        # h: optional initial recurrent hidden state

        h_seq, h_final = self.gru(x, h)

        # Gaussian at every time step
        mean, L = self.gaussian_head(h_seq)

        return mean, L, h_final

class RecurrentGaussianNetwork(nn.Module):
    def __init__(
        self,
        input_dim,
        output_dim,
        hidden_dim=64,
        num_layers=1,
        head_hidden_dims=(128,),
        min_std=1e-4,
        nonlinearity="tanh",
    ):
        super().__init__()

        # Vanilla recurrent network
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            nonlinearity=nonlinearity,
            batch_first=True,
        )

        # Gaussian output network
        self.gaussian_head = GaussianNetwork(
            input_dim=hidden_dim,
            output_dim=output_dim,
            hidden_dims=head_hidden_dims,
            min_std=min_std,
        )

    def forward(self, x, h=None):
        """
        Parameters
        ----------
        x : Tensor, shape (batch, time, input_dim)
            Input sequences.

        h : Tensor, optional
            Initial hidden state, shape
            (num_layers, batch, hidden_dim).

        Returns
        -------
        mean : Tensor, shape (batch, time, output_dim)
            Gaussian mean at each time step.

        L : Tensor, shape (batch, time, output_dim, output_dim)
            Cholesky factor at each time step.

        h_final : Tensor, shape (num_layers, batch, hidden_dim)
            Final recurrent hidden state.
        """

        # Process temporal sequence
        h_seq, h_final = self.rnn(x, h)

        # Gaussian parameters at every time step
        mean, L = self.gaussian_head(h_seq)

        return mean, L, h_final

def gaussian_kl_divergence_torch(
    true_mean,
    true_scale_tril,
    approx_mean,
    approx_scale_tril,
):
    true_dist = torch.distributions.MultivariateNormal(
        loc=true_mean,
        scale_tril=true_scale_tril,
    )

    approx_dist = torch.distributions.MultivariateNormal(
        loc=approx_mean,
        scale_tril=approx_scale_tril,
    )

    return torch.distributions.kl_divergence(
        true_dist,
        approx_dist,
    ) / torch.log(torch.tensor(2.0))

def I_XR_gaussian_mc(model, data_generator, N_RgX=1000, N_X=1000):
    I_XR = 0.0
    x = data_generator(N_X)
    mean, L, _ = model(x)
    time, output_size = mean.shape
    for t in range(time):
        f_RgX = torch.distributions.MultivariateNormal(
            loc=mean[t],
            scale_tril=L[t],
        )
        for _rgx in range(N_RgX):
            r_ = f_RgX.rsample()
            x_ = data_generator(N_X)
            mean_, L_, _ = model(x_)
            f_r = (1/N_X) * torch.sum(torch.distributions.MultivariateNormal(
                loc=mean_,
                scale_tril=L_,
            ).log_prob(r_).exp())
            I_XR

# def I_XR_gaussian_mc(mean, L, N_RgX=1000, N_X=1000):
#     time, output_size = mean.shape
#     I_XR = 0.0
#     for t in range(time):
#         f_RgX = torch.distributions.MultivariateNormal(
#             loc=mean[t],
#             scale_tril=L[t],
#         )
#         for _rgx in range(N_RgX):
#             r_ = f_RgX.rsample()
#             for _x in range(N_X):
